# Marso submission — colour-aware Diffusion Policy (run all)

This notebook trains and evaluates the **colour-aware** policy (`method=dp_rgb_color`) on
your fork, then commits the checkpoint + `submission.yaml` back to GitHub.

**Before running:** *Runtime -> Change runtime type -> T4 GPU*.

What this does, in order:
1. Clone your fork and install deps (pip, ~2 min).
2. Download the Kaggle competition demos (needs a Kaggle API token, one-time).
3. Train `dp_rgb_color` on the **hard** demos (~60-90 min on a T4).
4. Evaluate the checkpoint on **easy / medium / hard** and print the weighted score.
5. Show the best rollout video.
6. Commit the checkpoint + `submission.yaml` to your fork and push.

Background and rationale: see `MARSO_RUNBOOK.md` / `MARSO_WRITEUP.md` in the repo.


## 1. Install

In [ ]:
# CLEAN START: always clone fresh into /content so the path can never nest.
# WARNING: re-running this cell wipes the local copy (incl. any not-yet-committed checkpoint).
# Don't re-run it after training; only run it once at the start of a session.
import os, shutil
%cd /content
if os.path.exists('/content/berlin-marso-hackathon'):
    shutil.rmtree('/content/berlin-marso-hackathon')
!git clone https://github.com/patcadragos/berlin-marso-hackathon.git
%cd /content/berlin-marso-hackathon

!pip install mani-skill==3.0.1 diffusers==0.38.0 gymnasium torch torchvision hydra-core kagglehub -q
!pip install -e . -q

# Colab headless rendering (offscreen Vulkan/EGL)
os.environ['DISPLAY'] = ''
os.environ['PYOPENGL_PLATFORM'] = 'egl'

# Identify yourself for the commit at the end of this notebook
!git config user.email "you@example.com"
!git config user.name "patcadragos"


## 2. Kaggle demos

Join the competition first (Rules -> *I Understand and Accept*), then get an API token:
kaggle.com -> Settings -> API -> *Create New API Token*. Kaggle now issues a single token
like `KGAT_...` (shown once -- copy it). Paste it into `KAGGLE_TOKEN` below.


In [ ]:
import os

# Kaggle's NEW single API token (kaggle.com -> Settings -> API -> Create New API Token).
# Looks like "KGAT_...". Covers both auth mechanisms the client may check.
KAGGLE_TOKEN = "PASTE_YOUR_KAGGLE_API_TOKEN"

os.environ['KAGGLE_API_TOKEN'] = KAGGLE_TOKEN
kaggle_dir = os.path.expanduser('~/.kaggle')
os.makedirs(kaggle_dir, exist_ok=True)
token_path = os.path.join(kaggle_dir, 'access_token')
with open(token_path, 'w') as f:
    f.write(KAGGLE_TOKEN)
os.chmod(token_path, 0o600)

!python il/download_demos.py


### Watch training live with TensorBoard
Run this cell before the training cell — it auto-refreshes as metrics are written.


In [ ]:
%load_ext tensorboard
%tensorboard --logdir il/baselines/diffusion_policy/runs


## 3. Train the colour-aware policy (main run)

**We train on `easy` first, on purpose.** Easy = 2 parcels, fixed positions, no bin-swaps -> the
demos are clean and consistent, so behavior cloning can actually reproduce them and score points.
A `hard` run (6 parcels, bin-swaps, 200-step cap) repeatedly scored 0 here -- the classic
imitation-learning wall, not a bug. Easy is the reliable, bankable score; we can try medium next.

The colour-aware encoder + augmentation are still on (`method=dp_rgb_color`). ~1.5 h on a T4.


In [ ]:
# Main run: easy demos, colour-aware encoder. ~1.5 h on a T4.
# Checkpoint lands in runs/warehouse_rgb_dp_color/checkpoints/ (what eval + submission.yaml expect).
!python il/train.py method=dp_rgb_color demo_dir=easy flags.total_iters=20000 flags.eval_freq=4000

# Optional follow-up if easy works and you still have GPU time (run as a separate session):
# !python il/train.py method=dp_rgb_color demo_dir=medium flags.total_iters=20000 flags.eval_freq=4000


## 4. Evaluate on all three levels

Prints `SORT ACCURACY` for each level and computes the weighted `final_score`. Each run also
drops an mp4 under its `videos/` folder.


In [ ]:
import subprocess, re

CKPT = "il/baselines/diffusion_policy/runs/warehouse_rgb_dp_color/checkpoints/best_eval_sort_accuracy.pt"
assert os.path.exists(CKPT), f"checkpoint not found: {CKPT} -- did training finish?"

scores = {}
for level in ["easy", "medium", "hard"]:
    cmd = [
        "python", "eval.py", f"difficulty={level}",
        "policy=warehouse_sort.il_policy:load_dp_rgb_color",
        f"checkpoint={CKPT}", "eval_config=conf/eval/default.yaml",
    ]
    print(f"=== evaluating {level} ===")
    out = subprocess.run(cmd, capture_output=True, text=True)
    print(out.stdout[-3000:])
    if out.returncode != 0:
        print(out.stderr[-3000:])
    m = re.search(r"sort.?accuracy[:=]\s*([0-9.]+)", out.stdout, re.IGNORECASE)
    scores[level] = float(m.group(1)) if m else None

print("scores:", scores)
if all(v is not None for v in scores.values()):
    final = 0.2 * scores["easy"] + 0.3 * scores["medium"] + 0.5 * scores["hard"]
    print(f"final_score = 0.2*easy + 0.3*medium + 0.5*hard = {final:.4f}")
else:
    print("Could not auto-parse one or more SORT ACCURACY lines -- read them from the printed "
          "stdout above and compute final_score by hand.")


## 5. Watch the best rollout

Update `LEVEL` below to whichever level you want to inspect (try `hard` first — that's where
the colour-aware encoder should matter most).


In [ ]:
import glob
from IPython.display import Video, display

LEVEL = "hard"
vids = sorted(
    glob.glob(f"outputs/**/videos/*.mp4", recursive=True)
    + glob.glob(f"il/baselines/diffusion_policy/runs/*/videos/*.mp4"),
    key=os.path.getmtime,
)
if vids:
    print("playing:", vids[-1])
    display(Video(vids[-1], embed=True, width=640))
else:
    print("No eval video found yet.")


## 6. (Optional) A/B vs `plain_conv` fallback

Only run this if you have spare GPU time — compares the custom colour-aware encoder against
the zero-custom-code `plain_conv` fallback. Skip straight to **§7** if you're happy with the
colour-aware results above.


In [ ]:
# !python il/train.py method=dp_rgb demo_dir=hard #     flags.visual_encoder=plain_conv flags.aug_pad=4 flags.exp_name=warehouse_rgb_dp_plain
# # then eval with policy=warehouse_sort.il_policy:load_dp_rgb_plain


## 7. Commit the checkpoint + submission.yaml, push to your fork

This is what the judge actually clones and runs — make sure it lands on your fork's `main`.


In [ ]:
!git add il/baselines/diffusion_policy/runs/warehouse_rgb_dp_color/checkpoints/best_eval_sort_accuracy.pt submission.yaml
!git commit -m "Add trained colour-aware checkpoint"
!git push


## 8. Next: fill in the real numbers

Copy the `scores` printed in **§4** into `MARSO_WRITEUP.md` and `MARSO_VIDEO.md` (replace the
`[..]` placeholders), record your video from the rollout clips, then publish the Kaggle
Writeup with Project Link = your fork URL. Full checklist in `MARSO_RUNBOOK.md` Step 5.
